# Inverse Design: AI-Driven Composition Discovery for Perovskites

## 1. Introduction to Inverse Design
In traditional materials science, we follow a **Forward Approach**: 
*   `Composition` $\rightarrow$ `Experimental Testing` $\rightarrow$ `Properties`.

With **Inverse Design**, we reverse the process:
*   `Target Properties` (e.g., $PCE > 20\%$, $E_g = 1.55\ eV$) $\rightarrow$ `AI Search` $\rightarrow$ `Optimal Composition`.

This notebook uses the pre-trained XGBoost models and **Optuna** (a Bayesian optimization framework) to search the massive chemical space of perovskites.

---

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path to import our custom features logic
sys.path.append(os.path.abspath("../src"))

from ml_prediction_web_service.entities.dictionary import Element, Dimension, SpaceGroup, Site
from ml_prediction_web_service.services.features.calc_factors import compute_tolerance_factor, compute_octahedral_factor
from ml_prediction_web_service.services.features.structure_features import (
    calculate_effective_radii, 
    calculate_weighted_properties, 
    compute_dimensionality_indicator, 
    compute_space_group
)

print("Environment Ready.")

## 2. Load Trained ML Models

In [ ]:
band_gap_model_path = "../ml_models/xgboost_band_gap.joblib"
pce_t80_model_path = "../ml_models/pce_t80_model.joblib"

bg_model = joblib.load(band_gap_model_path)
pce_model = joblib.load(pce_t80_model_path)
print("Models loaded successfully.")

## 3. SCAPS-1D Parameter Estimator
After finding an optimal composition, we need to simulate it in SCAPS-1D to verify the device performance. This function estimates the physical constants needed for the simulation.

In [ ]:
def estimate_scaps_parameters(df_row):
    """
    Maps chemical properties to SCAPS-1D physical inputs.
    """
    bg = df_row['band_gap'].iloc[0]
    en_b = df_row['en_B'].iloc[0]
    
    # Heuristic for Electron Affinity (Chi) based on B-site electronegativity
    # Standard Pb-based perovskites have Chi ~ 3.9 eV
    chi = 3.9 + (en_b - 1.9) * 0.5
    
    # Dielectric constant (relative) - typical value for MAPbI3
    epsilon = 10.0 if df_row['is_2d'].iloc[0] == 0 else 6.5
    
    return {
        "Band Gap (Eg) [eV]": bg,
        "Electron Affinity (Chi) [eV]": round(chi, 2),
        "Dielectric Permittivity (epsilon)": epsilon,
        "CB Effective Density of States [cm-3]": "2.2E18",
        "VB Effective Density of States [cm-3]": "1.8E18",
        "Electron Mobility [cm2/Vs]": 20.0,
        "Hole Mobility [cm2/Vs]": 20.0
    }

## 4. Enhanced Objective Function with Physics Constraints

In [ ]:
TARGET_BAND_GAP = 1.55

def build_feature_row(a_sites, b_sites, c_sites):
    features = {"composition_inorganic": False}
    # Simplification for demo
    for i in range(1, 4):
        features[f"A_{i}"] = a_sites[i-1][0].nm if i <= len(a_sites) else "None"
        features[f"B_{i}"] = b_sites[i-1][0].nm if i <= len(b_sites) else "None"
        features[f"C_{i}"] = c_sites[i-1][0].nm if i <= len(c_sites) else "None"
    
    # Physics calcs
    class EF: 
        def __init__(self, e, f): self.name, self.frequence = e, f
    
    a_ef = [EF(e, f) for e, f in a_sites]
    b_ef = [EF(e, f) for e, f in b_sites]
    c_ef = [EF(e, f) for e, f in c_sites]
    
    r_A = calculate_effective_radii(a_ef)
    r_B = calculate_effective_radii(b_ef)
    r_C = calculate_effective_radii(c_ef)
    
    en_A, mass_A = calculate_weighted_properties(a_ef)
    en_B, mass_B = calculate_weighted_properties(b_ef)
    en_C, mass_C = calculate_weighted_properties(c_ef)
    
    t = compute_tolerance_factor(r_A, r_B, r_C)
    mu = compute_octahedral_factor(r_B, r_C)
    
    features.update({
        "r_A": r_A, "en_A": en_A, "mass_A": mass_A,
        "r_B": r_B, "en_B": en_B, "mass_B": mass_B,
        "r_C": r_C, "en_C": en_C, "mass_C": mass_C,
        "tolerance_factor": t, "octahedral_factor": mu,
        "is_2d": 0, "dimension": "3D", "space_group": "Cubic"
    })
    return pd.DataFrame([features])

def objective(trial):
    a1_name = trial.suggest_categorical("A_1", ["Cs", "MA", "FA"])
    f_a1 = trial.suggest_float("f_A1", 0.0, 1.0)
    a2_name = "Cs" if a1_name != "Cs" else "FA"
    
    a_sites = [(Element.get_element_by_name(a1_name), f_a1), (Element.get_element_by_name(a2_name), 1.0-f_a1)]
    b_sites = [(Element.get_element_by_name("Pb"), 1.0)]
    c_sites = [(Element.get_element_by_name("I"), 2.0), (Element.get_element_by_name("Br"), 1.0)]
    
    df_input = build_feature_row(a_sites, b_sites, c_sites)
    
    # --- PHYSICS CONSTRAINTS ---
    t = df_input["tolerance_factor"].iloc[0]
    mu = df_input["octahedral_factor"].iloc[0]
    
    # Goldschmidt limit for 3D perovskites
    if t < 0.8 or t > 1.05:
        return -999
    # Octahedral stability
    if mu < 0.4 or mu > 0.9:
        return -999
        
    bg_pred = bg_model.predict(df_input)[0]
    bg_error = abs(bg_pred - TARGET_BAND_GAP)
    
    return -bg_error # Maximize negative error (closer to 0 is better)

## 5. Run Search and Generate SCAPS Inputs

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(f"Best Parameters: {study.best_params}")

# Generate SCAPS report for best composition
best_a_sites = [(Element.get_element_by_name(study.best_params['A_1']), study.best_params['f_A1'])]
# ... (rebuild row)
best_row = build_feature_row(best_a_sites, [(Element.get_element_by_name("Pb"), 1.0)], [(Element.get_element_by_name("I"), 3.0)])
best_row['band_gap'] = bg_model.predict(best_row)[0]

print("\n--- SCAPS-1D INPUT PARAMETERS ---")
for k, v in estimate_scaps_parameters(best_row).items():
    print(f"{k}: {v}")